**Import relevant libraries**

In [24]:
import requests
import defeatbeta_api
from defeatbeta_api.data.ticker import Ticker
from defeatbeta_api.data.tickers import Tickers
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

**Define news class**

In [25]:
class Ticker_news:
    def __init__(self, ticker, headline, timestamp, source):
        self.ticker = ticker
        self.headline = headline
        self.timestamp = timestamp
        self.source = source

**Use defeatbeta-api API to pull news**

In [26]:
def get_data(tickers_file):
    with open(tickers_file, "r") as f:
        tickers = f.readline()
        tickers = tickers[1:-1]
        tickers = tickers.replace('"', "")
        tickers = tickers.split(",")
        tickers = [element.strip() for element in tickers]
        tickers_cl = Tickers(tickers)
    news = tickers_cl.news()
    #header_data = [{"Ticker": element, "Headline": news[element].get_news_list()["title"], "Timestamp": news[element].get_news_list()["report_date"], "Source": news[element].get_news_list()["publisher"]} for element in tickers]
    header_data = [Ticker_news(element, news[element].get_news_list()["title"], news[element].get_news_list()["report_date"], news[element].get_news_list()["publisher"]) for element in tickers]
    return header_data, tickers
    #print(header_data[0]["Source"])
    #print(news["TSLA"].get_news_list())
    #data = [{"Headline": news[element]["title"], }]
    #print(news["TSLA"].get_news_list()["title"])

**Analyze header probabilities**

In [27]:
def analyze_news_header_probabilities(news_data, tickers):
    data = {"Ticker": "", "Sentiment": "", "Polarity": ""}
    df_tickers = pd.DataFrame(data=data)
    analyzer = SentimentIntensityAnalyzer()

    for i in range(len(news_data)):
        scores = analyzer.polarity_scores(news_data[i].headline)
        polarity = scores['compound']
        if polarity > 0.05:
            sentiment = 'Positive'
        elif polarity < -0.05:
            sentiment = 'Negative'
        else:
            sentiment = 'Neutral'
        new_rows = pd.DataFrame({"Ticker": news_data[i].ticker, "Sentiment": sentiment, "Polarity": polarity})
        df_tickers = pd.concat([df_tickers, new_rows])

    return df_tickers

**Main program**

In [28]:
def main():
    #finbert_model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
    #finbert_tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
    #labels = ['Positive', 'Negative', 'Neutral']
    
    news_data, tickers = get_data("tickers.txt")
    #df_tickers = analyze_news_header_probabilities(news_data, tickers)
    print(news_data)
main()

[<__main__.Ticker_news object at 0x120b52270>, <__main__.Ticker_news object at 0x11c27e350>, <__main__.Ticker_news object at 0x11f25f110>, <__main__.Ticker_news object at 0x111824fc0>, <__main__.Ticker_news object at 0x111824770>, <__main__.Ticker_news object at 0x12402d490>, <__main__.Ticker_news object at 0x11f26e580>, <__main__.Ticker_news object at 0x11f26e690>, <__main__.Ticker_news object at 0x11f4aa350>, <__main__.Ticker_news object at 0x11f4aa250>, <__main__.Ticker_news object at 0x11be26d50>, <__main__.Ticker_news object at 0x11be273e0>, <__main__.Ticker_news object at 0x11cbe4830>, <__main__.Ticker_news object at 0x11cbe4f30>, <__main__.Ticker_news object at 0x120bbd160>, <__main__.Ticker_news object at 0x123faf590>, <__main__.Ticker_news object at 0x123faf650>, <__main__.Ticker_news object at 0x11cb58680>, <__main__.Ticker_news object at 0x11cb8bc20>, <__main__.Ticker_news object at 0x120bdb610>, <__main__.Ticker_news object at 0x120bdb6b0>, <__main__.Ticker_news object at 0